In [41]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupKFold,StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from tqdm import tqdm
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, matthews_corrcoef,roc_auc_score



# =========================
# Dataset Path
# =========================
feature_path = r"D:\M143020071\926\Xgboost_result\feature"
save_path = r"D:\M143020071\926\Xgboost_result\result\420+200\10times5fold_bal"


os.makedirs(save_path, exist_ok=True)


In [42]:
def get_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    denom = tp + tn + fp + fn
    acc = (tp + tn) / denom if denom > 0 else 0
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    mcc = matthews_corrcoef(y_true, y_pred)
    return [acc, sens, spec, mcc]

def evaluate_predictions(y_true, y_prob, groups, threshold=0.5):
    win_metrics = get_metrics(y_true, (y_prob >= threshold).astype(int))
    df = pd.DataFrame({'ID': groups, 'Actual': y_true, 'Prob': y_prob}).groupby('ID').mean()
    sub_metrics = get_metrics(df['Actual'], (df['Prob'] >= threshold).astype(int))
    return win_metrics, sub_metrics, df

def plot_confusion_matrix(y_true, y_pred, title, ax):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Non-MI(0)', 'MI(1)'], 
                yticklabels=['Non-MI(0)', 'MI(1)'], ax=ax)
    ax.set_title(title)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

def save_cm_to_csv(y_true, y_pred, filename):
    cm = confusion_matrix(y_true, y_pred)
    cm_df = pd.DataFrame(cm, 
                         index=['Actual_0', 'Actual_1'], 
                         columns=['Predicted_0', 'Predicted_1'])
    full_path = os.path.join(save_path, f"{filename}.csv")
    cm_df.to_csv(full_path)
    print(f"Saved Confusion Matrix: {full_path}")


def process_data(file_path):
    print(f"Opening {os.path.basename(file_path)} ...")
    
    # 直接讀取 .npz 字典
    data = np.load(file_path, allow_pickle=True)
    
    # 根據 np.savez 指定的 key 直接安全拿取
    X = data['X'].astype(np.float32)
    y = data['y'].astype(np.int8)
    groups = data['groups']
    
    print(f"Data Loaded! X: {X.shape}, y: {y.shape}, groups: {groups.shape}")
    print(f"Labels checking: {np.unique(y)}") 
    
    return X, y, groups 

In [43]:
hyper_params_dict = {
    'objective': 'binary:logistic',
    'booster': 'gbtree', 
    'eval_metric': 'aucpr', 
    'learning_rate': 0.05, 
    'n_estimators': 500, 
    'max_depth': 3, 
    'min_child_weight': 1,
    'gamma': 0.1, 
    'subsample': 0.8, 
    'colsample_bytree': 0.8, 
    'reg_alpha': 0.01, 
    'reg_lambda': 1, 
    'random_state': 42
}

In [44]:
feature_files = ["iSKNA_420+200.npz"] # 要改dataset名稱
all_final_metrics = []

n_iterations = 10
# 設定 10 個不同的隨機種子，確保每次 StratifiedGroupKFold 的隨機分配都不同
seeds = [42 + i for i in range(n_iterations)] 

for f_file in feature_files:
    print(f"\n{'='*40}\nProcessing Feature: {f_file}\n{'='*40}")
    
    file_path = os.path.join(feature_path, f_file)
    
    try:
        X, y, groups = process_data(file_path)
    except Exception as e:
        print(f"Error loading {f_file}: {e}")
        continue
    
    # 用來儲存這 10 次迭代的各組指標
    iter_win_metrics = []
    iter_sub_metrics = []
    
    for iter_idx, seed in enumerate(seeds):
        print(f"\n--- Iteration {iter_idx+1}/{n_iterations} (Seed: {seed}) ---")
        
        all_y_true = []
        all_y_probs = []
        all_groups = []
        
        # 使用 StratifiedGroupKFold 進行隨機分配，並帶入當前輪次的隨機種子
        sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
       
        for fold, (train_idx, test_idx) in enumerate(tqdm(sgkf.split(X, y, groups), desc=f"Iter {iter_idx+1} Folds")):
            
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            groups_test = groups[test_idx]
        
            # 1. 處理類別不平衡：動態計算此 fold 訓練集的負樣本與正樣本比例
            scale_weight = np.sum(y_train == 0) / np.sum(y_train == 1)

            # 2. 更改成 pipeline，並將 scale_pos_weight 傳入 XGBClassifier
            pipeline = make_pipeline(
                # StandardScaler(),
                # PCA(n_components=0.95, random_state=42), 
                XGBClassifier(**hyper_params_dict, scale_pos_weight=scale_weight)
            )
            
            pipeline.fit(X_train, y_train)
            
            test_prob = pipeline.predict_proba(X_test)[:, 1]
            
            # 將 y_prob 轉換為 yhead (超過 0.5 判定為 MI (1)，否則為 Non-MI (0))
            y_pred = (test_prob >= 0.5).astype(int)
            
            fold_details_df = pd.DataFrame({
                'Subject_ID': groups_test,     # 受試者 ID (檔名)
                'True_Label': y_test,          # 真正的 y (0 或 1)
                'Pred_Probability': test_prob, # 預測 Score (機率)
                'Pred_Label': y_pred           # 預測的 yhead (0 或 1)
            })
            
            # 建立專屬檔名並儲存 (檔名中加入 Iteration 與 Fold 資訊)
            fold_csv_name = f"{f_file.split('.')[0]}_Iter{iter_idx+1}_Fold{fold+1}_predictions.csv"
            fold_csv_path = os.path.join(save_path, fold_csv_name)
            fold_details_df.to_csv(fold_csv_path, index=False)
            
            all_y_true.append(y_test)
            all_y_probs.append(test_prob)
            all_groups.append(groups_test)
           
            # 釋放記憶體 (原本的 model 變數更新為 pipeline)
            del X_train, X_test, y_train, y_test, pipeline, fold_details_df
            gc.collect()

        # 5 折跑完一次後，把該輪 5 折的預測全部接起來進行整體評估
        y_true_combined = np.concatenate(all_y_true)
        y_probs_combined = np.concatenate(all_y_probs)
        groups_combined = np.concatenate(all_groups)

        win_metrics, sub_metrics, df_sub = evaluate_predictions(y_true_combined, y_probs_combined, groups_combined)
        
        # 記錄此輪算出來的一組指標
        iter_win_metrics.append(win_metrics)
        iter_sub_metrics.append(sub_metrics)

        # 繪製並儲存此輪的混淆矩陣圖片 (檔名加入 Iteration 資訊)
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        # plot_confusion_matrix(y_true_combined, (y_probs_combined >= 0.5).astype(int), f"Window-level CM (Iter {iter_idx+1})\n{f_file}", axes[0]) # 有win要開
        plot_confusion_matrix(df_sub['Actual'], (df_sub['Prob'] >= 0.5).astype(int), f"Subject-level CM (Iter {iter_idx+1})\n{f_file}", axes[1])
        
        plt.tight_layout()
        plt.savefig(os.path.join(save_path, f"{f_file.split('.')[0]}_Iter{iter_idx+1}_Stacked_CM.png"))
        plt.close() # 關閉畫布避免 Jupyter 畫面上殘留過多圖片導致記憶體溢出

        # 儲存此輪的混淆矩陣到 CSV (檔名加入 Iteration 資訊)
        # save_cm_to_csv(y_true_combined, (y_probs_combined >= 0.5).astype(int), f"{f_file.split('.')[0]}_Iter{iter_idx+1}_win_cm") # 有win要開
        save_cm_to_csv(df_sub['Actual'], (df_sub['Prob'] >= 0.5).astype(int), f"{f_file.split('.')[0]}_Iter{iter_idx+1}_sub_cm")

        res_prefix = f_file.split('.')[0]
        # 將每一輪的詳細指標都記錄到總清單中
        # all_final_metrics.append([f"{res_prefix}_Win_Iter{iter_idx+1}"] + win_metrics) # 有win要開
        all_final_metrics.append([f"{res_prefix}_Sub_Iter{iter_idx+1}"] + sub_metrics)

        del df_sub, y_true_combined, y_probs_combined, groups_combined
        gc.collect() 

    # ----------- 🎯 10 次重複動作全部完成，計算平均指標 -----------
    res_prefix = f_file.split('.')[0]
    
    # 計算 Window-level 的 10 次平均指標 (有win要開)
    # mean_win_metrics = np.mean(iter_win_metrics, axis=0)
    # all_final_metrics.append([f"{res_prefix}_Win_Average"] + list(mean_win_metrics))
    
    # 計算 Subject-level 的 10 次平均指標，並加入總表中
    mean_sub_metrics = np.mean(iter_sub_metrics, axis=0)
    all_final_metrics.append([f"{res_prefix}_Sub_Average"] + list(mean_sub_metrics))

    del X, y, groups
    gc.collect() 

# ----------- 儲存最終的評估指標總表 -----------
cols = ['Model_Type', 'Accuracy', 'Sensitivity', 'Specificity', 'MCC']
summary_df = pd.DataFrame(all_final_metrics, columns=cols)
summary_df.to_csv(os.path.join(save_path, "XGBoost_Final_ALL_Metrics.csv"), index=False)

print("\nProcessing Complete!")
print(summary_df)
print(save_path)


Processing Feature: iSKNA_420+200.npz
Opening iSKNA_420+200.npz ...
Data Loaded! X: (620, 3000), y: (620,), groups: (620,)
Labels checking: [0 1]

--- Iteration 1/10 (Seed: 42) ---


Iter 1 Folds: 0it [00:00, ?it/s]

Iter 1 Folds: 5it [01:09, 13.90s/it]


Saved Confusion Matrix: D:\M143020071\926\Xgboost_result\result\420+200\10times5fold_bal\iSKNA_420+200_Iter1_sub_cm.csv

--- Iteration 2/10 (Seed: 43) ---


Iter 2 Folds: 5it [01:16, 15.21s/it]


...

Saved Confusion Matrix: D:\M143020071\926\Xgboost_result\result\420+200\10times5fold_bal\iSKNA_420+200_Iter8_sub_cm.csv

--- Iteration 9/10 (Seed: 50) ---


Iter 9 Folds: 5it [01:28, 17.64s/it]


Saved Confusion Matrix: D:\M143020071\926\Xgboost_result\result\420+200\10times5fold_bal\iSKNA_420+200_Iter9_sub_cm.csv

--- Iteration 10/10 (Seed: 51) ---


Iter 10 Folds: 5it [01:23, 16.63s/it]


Saved Confusion Matrix: D:\M143020071\926\Xgboost_result\result\420+200\10times5fold_bal\iSKNA_420+200_Iter10_sub_cm.csv

Processing Complete!
                   Model_Type  Accuracy  Sensitivity  Specificity       MCC
0     iSKNA_420+200_Sub_Iter1  0.687097     0.866667        0.310  0.210379
1     iSKNA_420+200_Sub_Iter2  0.709677     0.883333        0.345  0.271904
2     iSKNA_420+200_Sub_Iter3  0.680645     0.861905        0.300  0.192800
3     iSKNA_420+200_Sub_Iter4  0.704839     0.883333        0.330  0.256569
4     iSKNA_420+200_Sub_Iter5  0.677419     0.857143        0.300  0.185934
5     iSKNA_420+200_Sub_Iter6  0.680645     0.852381        0.320  0.200254
6     iSKNA_420+200_Sub_Iter7  0.677419     0.859524        0.295  0.184010
7     iSKNA_420+200_Sub_Iter8  0.704839     0.888095        0.320  0.253736
8     iSKNA_420+200_Sub_Iter9  0.670968     0.857143        0.280  0.164387
9    iSKNA_420+200_Sub_Iter10  0.690323     0.866667        0.320  0.220867
10  iSKNA_420+200_Sub